In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_Res
from train import WarmUpCosine, CustomWeightDecaySGD
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_cla import load_wb_if_exists
from evaluate_cla import evalu_stream_main_selected, per_group_ovr_accuracy, evalu_select

In [4]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [5]:
model=load_Res()

In [6]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("re_lu_16").output,
    outputs=model.output
)

In [7]:
l_label=[3, 11, 18, 27, 34, 43, 50, 59, 66]

In [8]:
layer_list = [model.layers[l].name for l in l_label]

In [9]:
model.evaluate(x_train,y_train_onehot)

625/625 [==============================] - 6s 6ms/step - loss: 0.0329 - accuracy: 0.9998


[0.032918043434619904, 0.9998000264167786]

In [10]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_1/Res-18/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_1/Res-18/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_1/Res-18/layer_cache/salt/"+str(i))
    save_layer_outputs_and_labels(model, x_move, y_train, layer_list, save_dir="D:/Data_1/Res-18/layer_cache/move/"+str(i))
    save_layer_outputs_and_labels(model, x_occ, y_train, layer_list, save_dir="D:/Data_1/Res-18/layer_cache/occ/"+str(i))

[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/base
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/gauss/0
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/salt/0
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/move/0
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/occ/0
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/gauss/1
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/salt/1
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/move/1
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/occ/1
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/gauss/2
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/salt/2
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/move/2
[SKIP] All layer files already exist in D:/Data_1/Res-18/layer_cache/occ/2
[SKIP] All lay

In [11]:
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "Res_pgd.npy")
    x_attack = np.load(ATTACK_DIR)
    save_layer_outputs_and_labels(model, x_attack, y_train, layer_list, save_dir="D:/Data_1/Res-18/layer_cache/attack/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_1/Res-18/layer_cache/attack/0
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_6: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_8: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_10: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_12: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_14: outputs (20000, 1024), labels (20000, 1)
[Saved] re_lu_16: outputs (20000, 1024), labels (20000, 1)
[SAVE] Generating layer outputs for: D:/Data_1/Res-18/layer_cache/attack/1
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_6: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_8: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_10: outputs (20000, 20

In [12]:
CACHE_DIR = "./Res-18/w_and_b_cache"

In [13]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_1/Res-18/layer_cache/base")

In [14]:
NOISE_dir='./noise/'
ATTACK_dir=Noise_dir='./attack/'

In [15]:
pred_model.input_shape

(None, 2, 2, 256)

In [16]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_1/Res-18/layer_cache/base")

[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]


In [17]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_1/Res-18/layer_cache/base")
base_final = per_group_ovr_accuracy(model, x_train, y_train, select_list)
base = np.concatenate((base,base_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.66579449 0.63567635 0.63226672 0.58462358]
Layer 1
accuracy: [0.68461766 0.70150061 0.64709347 0.56001595]
Layer 2
accuracy: [0.71125045 0.69826395 0.68411493 0.58272164]
Layer 3
accuracy: [0.75207547 0.74426112 0.74358175 0.56800164]
Layer 4
accuracy: [0.76981568 0.80275085 0.74812569 0.64656003]
Layer 5
accuracy: [0.88646715 0.93889721 0.90463068 0.90801683]
Layer 6
accuracy: [0.97554299 0.98895456 0.97183686 0.9842162 ]
Layer 7
accuracy: [0.98199404 0.99200935 0.98796528 0.9854799 ]
Layer 8
accuracy: [0.98950999 0.99449207 0.9934749  0.98799917]


In [18]:
base

array([[0.66579449, 0.63567635, 0.63226672, 0.58462358],
       [0.68461766, 0.70150061, 0.64709347, 0.56001595],
       [0.71125045, 0.69826395, 0.68411493, 0.58272164],
       [0.75207547, 0.74426112, 0.74358175, 0.56800164],
       [0.76981568, 0.80275085, 0.74812569, 0.64656003],
       [0.88646715, 0.93889721, 0.90463068, 0.90801683],
       [0.97554299, 0.98895456, 0.97183686, 0.9842162 ],
       [0.98199404, 0.99200935, 0.98796528, 0.9854799 ],
       [0.98950999, 0.99449207, 0.9934749 , 0.98799917],
       [1.        , 1.        , 1.        , 1.        ]])

In [19]:
attack = np.zeros((len(layer_list),4))
for i in range(5):
    attack += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Res-18/layer_cache/attack/"+str(i))
attack = attack/5
attack_final = np.zeros(4)
for i in range(5):
    DIR = ATTACK_dir + str(i) + '/Res_pgd.npy'
    x_noise = np.load(DIR)
    attack_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
attack_final = attack_final/5
attack = np.concatenate((attack,attack_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.64402854 0.58208932 0.61877758 0.53889325]
Layer 1
accuracy: [0.6711796  0.62063754 0.61664188 0.41028275]
Layer 2
accuracy: [0.72842631 0.64085404 0.66463294 0.3938385 ]
Layer 3
accuracy: [0.75663662 0.69007633 0.72756504 0.28560314]
Layer 4
accuracy: [0.71137396 0.65136365 0.69883225 0.2271513 ]
Layer 5
accuracy: [0.58671934 0.38887565 0.66292369 0.1768261 ]
Layer 6
accuracy: [0.55099978 0.4151663  0.55078777 0.16430432]
Layer 7
accuracy: [0.5497843  0.38911927 0.54601712 0.16306583]
Layer 8
accuracy: [0.55850324 0.45521575 0.54805499 0.17506551]
Layer 0
accuracy: [0.64502038 0.58136105 0.61829005 0.53989781]
Layer 1
accuracy: [0.67297879 0.61637861 0.61328123 0.40952683]
Layer 2
accuracy: [0.72699299 0.64058084 0.65790535 0.39528849]
Layer 3
accuracy: [0.75687867 0.69348837 0.72298213 0.27860392]
Layer 4
accuracy: [0.70821349 0.6526536  0.7028567  0.22084646]
Layer 5
accuracy: [0.58196972 0.39766769 0.66530557 0.1702606 ]
Layer 6
accuracy: [0.5511479  0.41579543

In [20]:
attack

array([[0.64477456, 0.5814797 , 0.6180485 , 0.53991759],
       [0.67259486, 0.61864825, 0.61398985, 0.41044226],
       [0.72863722, 0.63985882, 0.66181729, 0.39260329],
       [0.75596565, 0.69137269, 0.72569486, 0.28184748],
       [0.71040034, 0.65174844, 0.70166082, 0.22277328],
       [0.58444484, 0.39150372, 0.66556571, 0.17178602],
       [0.55139397, 0.41055833, 0.5523936 , 0.15937692],
       [0.54997964, 0.38851562, 0.54709447, 0.15903697],
       [0.55632846, 0.4548278 , 0.55103028, 0.17174289],
       [0.54945   , 0.48945   , 0.55074999, 0.16      ]])

In [21]:
gauss = np.zeros((len(layer_list),4))
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Res-18/layer_cache/gauss/"+str(i))
gauss = gauss/10
gauss_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/gauss.npy'
    x_noise = np.load(DIR)
    gauss_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.28243239 0.24999773 0.27108228 0.2503052 ]
Layer 1
accuracy: [0.26079217 0.24999773 0.24976213 0.25005816]
Layer 2
accuracy: [0.29801912 0.24999773 0.25025188 0.25005816]
Layer 3
accuracy: [0.74998578 0.71539705 0.74974812 0.46506755]
Layer 4
accuracy: [0.74999498 0.4539115  0.74949912 0.25207153]
Layer 5
accuracy: [0.74998578 0.75348353 0.74974812 0.3700785 ]
Layer 6
accuracy: [0.75022732 0.75201561 0.74999108 0.5425057 ]
Layer 7
accuracy: [0.75047757 0.75252204 0.74578081 0.28557881]
Layer 8
accuracy: [0.75022008 0.75226021 0.74529431 0.31881402]
Layer 0
accuracy: [0.28045787 0.24999773 0.27083865 0.25005816]
Layer 1
accuracy: [0.25855582 0.24999773 0.24953828 0.25005816]
Layer 2
accuracy: [0.29425742 0.24999773 0.25025188 0.25005816]
Layer 3
accuracy: [0.74998578 0.7141396  0.74974812 0.47329459]
Layer 4
accuracy: [0.75022732 0.44398195 0.75001018 0.25231561]
Layer 5
accuracy: [0.74998578 0.75227354 0.74974812 0.36853707]
Layer 6
accuracy: [0.75022732 0.75050772

In [22]:
gauss

array([[0.28219127, 0.24999773, 0.27222809, 0.25020638],
       [0.25968263, 0.24999773, 0.24969928, 0.25005816],
       [0.29773043, 0.24999773, 0.25025188, 0.25005816],
       [0.74998578, 0.71483548, 0.74974812, 0.47103951],
       [0.75014054, 0.44604753, 0.7497386 , 0.25223317],
       [0.74998578, 0.7524837 , 0.74974812, 0.36635637],
       [0.75025235, 0.75124551, 0.74979711, 0.53731258],
       [0.75065789, 0.75103233, 0.74542317, 0.28264699],
       [0.75048213, 0.75296658, 0.74399953, 0.31414257],
       [0.75082501, 0.7506    , 0.7439    , 0.34055001]])

In [23]:
salt = np.zeros((len(layer_list),4))
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Res-18/layer_cache/salt/"+str(i))
salt = salt/10
salt_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/salt.npy'
    x_noise = np.load(DIR)
    salt_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.37109372 0.25372847 0.34181323 0.25356212]
Layer 1
accuracy: [0.25123986 0.24999773 0.25099284 0.25005816]
Layer 2
accuracy: [0.255259   0.24999773 0.25174848 0.25005816]
Layer 3
accuracy: [0.66486513 0.2517512  0.70502489 0.25031586]
Layer 4
accuracy: [0.52359766 0.25175413 0.69540009 0.25030717]
Layer 5
accuracy: [0.7669185  0.77172374 0.74999108 0.72886735]
Layer 6
accuracy: [0.74928912 0.79139872 0.75124089 0.78605651]
Layer 7
accuracy: [0.66087371 0.80482026 0.76229466 0.74085561]
Layer 8
accuracy: [0.69499693 0.81515725 0.75828607 0.73908536]
Layer 0
accuracy: [0.36815844 0.25423391 0.35313355 0.25257315]
Layer 1
accuracy: [0.25077322 0.24999773 0.25049483 0.25005816]
Layer 2
accuracy: [0.25524255 0.24999773 0.25075305 0.25005816]
Layer 3
accuracy: [0.66216812 0.25146565 0.70585903 0.25131336]
Layer 4
accuracy: [0.5219675  0.25247947 0.69492426 0.2503052 ]
Layer 5
accuracy: [0.77133031 0.77071738 0.74949912 0.73286107]
Layer 6
accuracy: [0.74857579 0.79422874

In [24]:
salt

array([[0.37240335, 0.2543607 , 0.35121397, 0.2520559 ],
       [0.25086175, 0.24999773, 0.25044822, 0.25005816],
       [0.25550004, 0.24999773, 0.25119366, 0.25005816],
       [0.66019842, 0.25307205, 0.70916365, 0.25040782],
       [0.5261849 , 0.25225519, 0.69158914, 0.25013267],
       [0.76742128, 0.76953222, 0.74977242, 0.7294247 ],
       [0.74657177, 0.79505207, 0.75096728, 0.78446858],
       [0.65874796, 0.80084881, 0.76178217, 0.73581975],
       [0.69787286, 0.82092248, 0.75958169, 0.73516916],
       [0.67475001, 0.82747499, 0.766325  , 0.756525  ]])

In [25]:
move = np.zeros((len(layer_list),4))
for i in range(10):
    move += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Res-18/layer_cache/move/"+str(i))
move = move/10
move_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/move.npy'
    x_noise = np.load(DIR)
    move_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
move_final = move_final/10
move = np.concatenate((move,move_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.74618293 0.74827253 0.7390528  0.74744506]
Layer 1
accuracy: [0.75001385 0.74975474 0.74604132 0.74994184]
Layer 2
accuracy: [0.75080571 0.74975474 0.74749493 0.74994184]
Layer 3
accuracy: [0.73569182 0.74975572 0.77839107 0.7499443 ]
Layer 4
accuracy: [0.74981559 0.75000227 0.78515857 0.74994184]
Layer 5
accuracy: [0.63295859 0.75000227 0.67794495 0.74969121]
Layer 6
accuracy: [0.6901295  0.75123501 0.49955201 0.71339853]
Layer 7
accuracy: [0.70995895 0.75024882 0.74964635 0.74717194]
Layer 8
accuracy: [0.70476353 0.75000227 0.69875278 0.74570285]
Layer 0
accuracy: [0.74624496 0.74876758 0.73902544 0.74820569]
Layer 1
accuracy: [0.7489809  0.74975474 0.74748667 0.74994184]
Layer 2
accuracy: [0.74853837 0.74975474 0.74721915 0.74994184]
Layer 3
accuracy: [0.73322071 0.75000227 0.77631108 0.74968726]
Layer 4
accuracy: [0.7475174  0.75000227 0.78815213 0.74994184]
Layer 5
accuracy: [0.63420326 0.75026214 0.67303771 0.74840448]
Layer 6
accuracy: [0.68583282 0.75102844

In [26]:
move

array([[0.74535588, 0.7486658 , 0.73884124, 0.74772218],
       [0.74917319, 0.74975474, 0.74659804, 0.74994184],
       [0.74867465, 0.74980548, 0.74724611, 0.74994184],
       [0.73434492, 0.74970059, 0.77615143, 0.74991535],
       [0.75012515, 0.74997628, 0.78827125, 0.74994184],
       [0.62901224, 0.75017648, 0.67223   , 0.74808013],
       [0.69025544, 0.75082664, 0.50007252, 0.71156177],
       [0.71074102, 0.75040104, 0.74291634, 0.74483251],
       [0.7061022 , 0.75027377, 0.69935314, 0.7442655 ],
       [0.60294999, 0.752975  , 0.583675  , 0.7214    ]])

In [27]:
occ = np.zeros((len(layer_list),4))
for i in range(10):
    occ += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Res-18/layer_cache/occ/"+str(i))
occ = occ/10
occ_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/occ.npy'
    x_noise = np.load(DIR)
    occ_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
occ_final = occ_final/10
occ = np.concatenate((occ,occ_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.59097227 0.56580507 0.56604441 0.48293535]
Layer 1
accuracy: [0.65495807 0.68486159 0.62759441 0.54703203]
Layer 2
accuracy: [0.66383233 0.67570564 0.64384055 0.52503798]
Layer 3
accuracy: [0.71573878 0.71117907 0.74301262 0.45628771]
Layer 4
accuracy: [0.7222247  0.74850414 0.74180614 0.55949934]
Layer 5
accuracy: [0.80584547 0.7972065  0.82933579 0.77980443]
Layer 6
accuracy: [0.82758076 0.81041855 0.85951727 0.82022758]
Layer 7
accuracy: [0.83450233 0.82712323 0.8601445  0.71687996]
Layer 8
accuracy: [0.8335855  0.82786483 0.84716456 0.69634539]
Layer 0
accuracy: [0.58751069 0.56319431 0.56559323 0.4825004 ]
Layer 1
accuracy: [0.65693915 0.68875501 0.62460197 0.53399977]
Layer 2
accuracy: [0.66717071 0.67199214 0.64745653 0.5235128 ]
Layer 3
accuracy: [0.71708452 0.71332488 0.74105441 0.45622883]
Layer 4
accuracy: [0.71734684 0.75078126 0.74378659 0.54403586]
Layer 5
accuracy: [0.79983686 0.79272084 0.82495824 0.76749377]
Layer 6
accuracy: [0.8151365  0.80451828

In [28]:
occ

array([[0.58566805, 0.5647036 , 0.56597339, 0.4803209 ],
       [0.65343668, 0.68933521, 0.62628101, 0.5372772 ],
       [0.66431915, 0.6753105 , 0.650276  , 0.52224224],
       [0.71285337, 0.7152572 , 0.74220853, 0.45507789],
       [0.71925747, 0.75243188, 0.74362238, 0.54435414],
       [0.80072329, 0.79677143, 0.82847278, 0.77097407],
       [0.81810478, 0.80984853, 0.86113381, 0.80556282],
       [0.82386723, 0.82782924, 0.85948238, 0.70821858],
       [0.82172206, 0.82768641, 0.84335594, 0.68767748],
       [0.86492501, 0.84825   , 0.88565   , 0.74267501]])

In [29]:
SAVE_FILE='Res-18.pkl'

In [30]:
progress = {"base": base, "attack": attack,"gauss": gauss,"salt": salt,"move": move,"occ": occ}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [31]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=6):
    """
    Input:
        matrix_dict: {name: np.ndarray(m, n)}, 6 matrices total
    Output:
        stats: {
            name: {
                "mean": (m,),
                "min":  (m,),
                "max":  (m,)
            }, ...
        }
    For each component i, compute statistics along axis=1 (across n samples):
        mean[i] = mean(X[i, :])
        min[i]  = min(X[i, :])
        max[i]  = max(X[i, :])
    """
    if check_num is not None and len(matrix_dict) != check_num:
        raise ValueError(f"Expected {check_num} matrices, got {len(matrix_dict)}")

    # shape consistency
    shapes = [np.asarray(v).shape for v in matrix_dict.values()]
    if len(set(shapes)) != 1:
        raise ValueError(f"Matrix shapes inconsistent: {shapes}")

    stats = {}
    for name, X in matrix_dict.items():
        X = np.asarray(X)
        stats[name] = {
            "mean": X.mean(axis=1),
            "std":  X.std(axis=1),
            "min":  X.min(axis=1),
            "max":  X.max(axis=1),
        }
    return stats

In [33]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [34]:
mean_var

{'base': {'mean': array([0.62959029, 0.64830692, 0.66908774, 0.70198   , 0.74181306,
         0.90950297, 0.98013765, 0.98686214, 0.99136903, 1.        ]),
  'std': array([0.02905578, 0.05464615, 0.0507786 , 0.07742441, 0.05833213,
         0.01884554, 0.00678945, 0.00365102, 0.00269259, 0.        ]),
  'min': array([0.58462358, 0.56001595, 0.58272164, 0.56800164, 0.64656003,
         0.88646715, 0.97183686, 0.98199404, 0.98799917, 1.        ]),
  'max': array([0.66579449, 0.70150061, 0.71125045, 0.75207547, 0.80275085,
         0.93889721, 0.98895456, 0.99200935, 0.99449207, 1.        ])},
 'attack': {'mean': array([0.59605509, 0.57891881, 0.60572915, 0.61372017, 0.57164572,
         0.45332507, 0.41843071, 0.41115668, 0.43348236, 0.4374125 ]),
  'std': array([0.03943716, 0.09995994, 0.12731824, 0.1929647 , 0.20266055,
         0.19060732, 0.16030917, 0.15955239, 0.15642239, 0.16206741]),
  'min': array([0.53991759, 0.41044226, 0.39260329, 0.28184748, 0.22277328,
         0.17178602, 